# Day 16 - Hypothesis Testing (Hands-On Activity)

This notebook walks through the **complete hypothesis-testing workflow** on a
business scenario, covering both a **numeric comparison** and a **categorical
relationship**.

The workflow has eight steps:

1. Load the dataset
2. Define the business question
3. Formulate the hypotheses
4. Choose the test
5. Execute the tests
6. Interpret the p-values
7. Make the decision
8. Document the findings

Two scenarios are analysed throughout:

- **Scenario A (numeric):** Do two stores have different average sales?
- **Scenario B (categorical):** Is a customer's region related to product preference?

## Step 1 - Load the dataset

**Objective:** Create the data for two scenarios - a numeric comparison and a
categorical relationship.

- **Scenario A** holds the weekly sales of two stores (two separate groups of numbers).
- **Scenario B** is a contingency table: rows are regions (North, South) and
  columns are preferred product categories (Electronics, Clothing, Groceries).

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency

# Scenario A: sales of two stores (numeric comparison)
store_a = [120, 135, 128, 142, 138, 150, 145, 133]
store_b = [110, 118, 122, 115, 120, 125, 119, 121]

# Scenario B: region versus preferred product (categorical relationship)
# Columns: Electronics, Clothing, Groceries
# Row 0 = North, Row 1 = South
contingency = np.array([
    [35, 25, 10],   # North
    [15, 30, 25],   # South
])

print("Data prepared.")
print("Store A:", store_a)
print("Store B:", store_b)
print("Contingency table (rows=region, cols=Electronics/Clothing/Groceries):")
print(contingency)

Data prepared.
Store A: [120, 135, 128, 142, 138, 150, 145, 133]
Store B: [110, 118, 122, 115, 120, 125, 119, 121]
Contingency table (rows=region, cols=Electronics/Clothing/Groceries):
[[35 25 10]
 [15 30 25]]


## Step 2 - Define the business question

- **Scenario A:** "Do Store A and Store B have significantly different average sales?"
- **Scenario B:** "Is a customer's region related to their preferred product category?"

## Step 3 - Formulate the hypotheses

Hypotheses must be stated **before** looking at the results.

**Scenario A**
- H0 (null): the two stores have equal average sales.
- H1 (alternative): the two stores' average sales differ.

**Scenario B**
- H0 (null): region and product preference are independent.
- H1 (alternative): region and product preference are related.

Significance level: **alpha = 0.05**.

## Step 4 - Choose the test

- **Scenario A** compares the averages of **two separate groups**, so an
  **independent (two-sample) t-test** (`stats.ttest_ind`) is appropriate.
- **Scenario B** examines a relationship between **two categorical variables**,
  so a **chi-square test of independence** (`chi2_contingency`) is appropriate.

## Step 5 - Execute the tests

In [2]:
alpha = 0.05

# Scenario A: independent t-test
t_stat, p_a = stats.ttest_ind(store_a, store_b)

# Scenario B: chi-square test of independence
chi2, p_b, dof, expected = chi2_contingency(contingency)

print("Scenario A t-statistic:", round(t_stat, 4))
print("Scenario A p-value:", round(p_a, 6))
print()
print("Scenario B chi-square:", round(chi2, 4))
print("Scenario B degrees of freedom:", dof)
print("Scenario B p-value:", round(p_b, 6))

Scenario A t-statistic: 4.6826
Scenario A p-value: 0.000353

Scenario B chi-square: 14.8831
Scenario B degrees of freedom: 2
Scenario B p-value: 0.000586


**Expected output (verified by running this notebook):**

```
Scenario A t-statistic: 4.6826
Scenario A p-value: 0.000353

Scenario B chi-square: 14.8831
Scenario B degrees of freedom: 2
Scenario B p-value: 0.000586
```

> **Note:** Some printed materials quote `Scenario A p = 0.000093` and
> `Scenario B p = 0.003` for this exact data. Those figures do not match what
> `ttest_ind` / `chi2_contingency` actually return - the verified values are
> `0.000353` and `0.000586`. Both are well below 0.05, so every conclusion below
> is unchanged.

## Step 6 - Interpret the p-values

Compare each p-value to `alpha = 0.05`. The cell below applies the decision rule
automatically so the interpretation always matches the computed result.

In [3]:
mean_a = sum(store_a) / len(store_a)
mean_b = sum(store_b) / len(store_b)

print("Store A average sales:", round(mean_a, 1))
print("Store B average sales:", round(mean_b, 1))
print()
print("Scenario A:", "Significant" if p_a < alpha else "Not significant",
      "(p =", round(p_a, 6), ")")
print("Scenario B:", "Significant" if p_b < alpha else "Not significant",
      "(p =", round(p_b, 6), ")")

Store A average sales: 136.4
Store B average sales: 118.8

Scenario A: Significant (p = 0.000353 )
Scenario B: Significant (p = 0.000586 )


Both p-values are below 0.05.

- **Scenario A is significant:** the two stores differ in average sales.
- **Scenario B is significant:** region and product preference are related.

## Step 7 - Make the decision

- **Scenario A:** Reject H0. Store A genuinely outperforms Store B (about 136 vs
  about 119 in average sales), which justifies investigating what drives the
  difference.
- **Scenario B:** Reject H0. Product preference depends on region, which
  justifies region-specific stocking.

## Step 8 - Document the findings

- Store A's average sales (about 136) are **significantly higher** than Store B's
  (about 119); p = 0.000353.
  **Recommendation:** investigate and replicate Store A's practices.
- Region and product preference are **significantly related**; p = 0.000586.
  **Recommendation:** tailor the product assortment by region.

Having completed all eight steps, the full hypothesis-testing workflow - from
business question to formal decision - has now been executed for both a numeric
comparison and a categorical relationship.

## Industry Best Practices

- **Always define the hypotheses first.** State H0 and H1, and set the
  significance level, *before* examining the data. Deciding what to test after
  seeing the results invalidates the test.
- **Understand the business context.** A hypothesis test serves a business
  question; frame the hypotheses to answer that question precisely.
- **Choose the correct test.** Match the test to the data type and comparison
  (separate groups -> independent t-test; same subjects twice -> paired t-test;
  two categorical variables -> chi-square).
- **Validate assumptions.** Tests rest on assumptions (such as adequate sample
  size). For introductory analysis, a reasonable sample size and the right data
  type are the main concerns.
- **Consider effect size alongside significance.** Significance says the effect
  is real; effect size says whether it is large enough to matter commercially.
  Report both.
- **Document conclusions clearly.** Record the hypotheses, the test used, the
  p-value, the decision, and the recommendation, so the analysis is transparent
  and reproducible.
- **Avoid overinterpreting results.** State conclusions in proportion to the
  evidence, distinguish correlation from causation, and never claim more than the
  data supports.
- **Treat hypothesis testing as a decision aid, not an oracle.** It informs the
  decision; it does not replace business judgement about costs, risks, and
  context.